# Camada Silver — Movimentos

Lê a Bronze, limpa e estrutura os dados:
- Achata o struct `_source` em colunas planas
- Explode o array `movimentos` (1 linha por evento processual)
- Parseia o número CNJ em segmentos separados
- Calcula campos temporais com window functions
- Grava em `judicial.silver.movimentos` via MERGE incremental

**Fonte:** `judicial.bronze.datajud_raw`  
**Destino:** `judicial.silver.movimentos`

## 1. Criar schema Silver

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS judicial.silver;

## 2. Parâmetros

In [ ]:
BRONZE_TABLE = "judicial.bronze.datajud_raw"
SILVER_TABLE = "judicial.silver.movimentos"

## 3. Leitura e achatamento da Bronze

A Bronze guarda cada hit do Elasticsearch com `_source` como struct aninhado.
Aqui extraímos os campos relevantes em colunas planas e explodimos
o array `movimentos` — cada evento processual vira uma linha independente.

In [ ]:
from pyspark.sql.functions import (
    col, explode, to_timestamp,
    regexp_extract,
    datediff, lag, row_number, desc,
    current_timestamp,
)
from pyspark.sql import Window

df_bronze = spark.table(BRONZE_TABLE)

df_movimentos = (
    df_bronze
    .select(
        col("_source.numeroProcesso").alias("numero_processo"),
        col("_source.tribunal").alias("sigla_tribunal"),
        col("_source.grau").alias("grau"),
        to_timestamp(col("_source.dataAjuizamento")).alias("data_ajuizamento"),
        col("_source.classe.codigo").alias("codigo_classe"),
        col("_source.classe.nome").alias("nome_classe"),
        col("_source.orgaoJulgador.codigo").alias("codigo_orgao"),
        col("_source.orgaoJulgador.nome").alias("nome_orgao"),
        col("_source.nivelSigilo").alias("nivel_sigilo"),
        explode(col("_source.movimentos")).alias("movimento"),
    )
    .select(
        "numero_processo",
        "sigla_tribunal",
        "grau",
        "data_ajuizamento",
        "codigo_classe",
        "nome_classe",
        "codigo_orgao",
        "nome_orgao",
        "nivel_sigilo",
        col("movimento.codigo").alias("codigo_movimento"),
        col("movimento.nome").alias("nome_movimento"),
        to_timestamp(col("movimento.dataHora")).alias("data_movimento"),
    )
)

print(f"Movimentos extraídos: {df_movimentos.count():,}")

## 4. Parse do número CNJ

A API retorna o número como 20 dígitos consecutivos, sem separadores.
Formato: `NNNNNNN DD AAAA J TT OOOO`
- `NNNNNNN` — número sequencial no órgão (7 dígitos)
- `DD` — dígitos verificadores (2 dígitos)
- `AAAA` — ano de ajuizamento (4 dígitos)
- `J` — segmento de justiça: 1=STF, 2=CNJ, 3=STJ, 4=JF, 5=TJ, 6=TRT, 7=TRE, 8=TJM (1 dígito)
- `TT` — código do tribunal (2 dígitos)
- `OOOO` — código do órgão de origem/vara (4 dígitos)

Exemplo: `00002464420248160192` → `0000246` `44` `2024` `8` `16` `0192`

In [ ]:
# 20 dígitos consecutivos — sem separadores
CNJ_PATTERN = r"(\d{7})(\d{2})(\d{4})(\d)(\d{2})(\d{4})"

df_parsed = (
    df_movimentos
    .withColumn("cnj_sequencial",   regexp_extract("numero_processo", CNJ_PATTERN, 1))
    .withColumn("cnj_digitos",      regexp_extract("numero_processo", CNJ_PATTERN, 2))
    .withColumn("cnj_ano",          regexp_extract("numero_processo", CNJ_PATTERN, 3).cast("int")) #Ajuda na filtragem de query
    .withColumn("cnj_justica",      regexp_extract("numero_processo", CNJ_PATTERN, 4).cast("int"))
    .withColumn("cnj_cod_tribunal", regexp_extract("numero_processo", CNJ_PATTERN, 5).cast("int"))
    .withColumn("cnj_origem",       regexp_extract("numero_processo", CNJ_PATTERN, 6))
)

## 5. Window functions

Window functions operam sobre uma "janela" de linhas relacionadas — aqui, todos os
movimentos do mesmo processo, ordenados por data.

- `lag(data_movimento)` devolve a data do movimento anterior na mesma janela
- `datediff` calcula a diferença em dias entre duas datas
- `row_number` numera as linhas dentro da janela — número 1 = movimento mais recente

In [ ]:
# Dois specs de janela: crescente (para lag/datediff) e decrescente (para is_ultimo)
w_asc  = Window.partitionBy("numero_processo").orderBy("data_movimento")
w_desc = Window.partitionBy("numero_processo").orderBy(desc("data_movimento"))

df_final = (
    df_parsed
    .withColumn(
        "dias_desde_ajuizamento",
        datediff("data_movimento", "data_ajuizamento")
    )
    .withColumn(
        "dias_desde_ultimo_movimento",
        # null no primeiro movimento de cada processo — correto, não há anterior
        # lag() pega a data do movimento que veio antes sobre a window w_asc.
        datediff("data_movimento", lag("data_movimento").over(w_asc))
    )
    .withColumn(
        "is_ultimo_movimento",
        row_number().over(w_desc) == 1
    )
    .withColumn("data_processamento", current_timestamp())
)

df_final.select(
    "numero_processo", "data_ajuizamento", "nome_movimento",
    "data_movimento", "dias_desde_ajuizamento",
    "dias_desde_ultimo_movimento", "is_ultimo_movimento"
).orderBy("numero_processo", "data_movimento").show(20, truncate=False)

## 6. MERGE incremental na Silver

A chave de match é a combinação `(numero_processo, codigo_movimento, data_movimento)`:
identifica univocamente um evento processual.

- `whenMatchedUpdateAll` — atualiza o registro se algum campo mudou (ex: `is_ultimo_movimento` virou False)
- `whenNotMatchedInsertAll` — insere movimentos novos que ainda não existem na Silver

In [ ]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists(SILVER_TABLE):
    # Primeira carga: cria a tabela com particionamento
    (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("sigla_tribunal", "cnj_ano")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Tabela criada: {SILVER_TABLE}")
else:
    dt = DeltaTable.forName(spark, SILVER_TABLE)
    (
        dt.alias("target")
        .merge(
            df_final.alias("source"),
            """
            target.numero_processo  = source.numero_processo
            AND target.codigo_movimento = source.codigo_movimento
            AND target.data_movimento   = source.data_movimento
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE concluído em {SILVER_TABLE}")

## 7. Verificação

In [ ]:
%sql
-- Contagem total vs distinta: se total_movimentos == chave_unica, não há duplicatas
SELECT
    COUNT(*)                                                                          AS total_movimentos,
    COUNT(DISTINCT numero_processo)                                                   AS processos_unicos,
    COUNT(DISTINCT CONCAT(numero_processo, codigo_movimento, CAST(data_movimento AS STRING))) AS chave_unica
FROM judicial.silver.movimentos;

In [ ]:
%sql
-- Valida que dias_desde_ajuizamento nunca é negativo
SELECT COUNT(*) AS dias_negativos
FROM judicial.silver.movimentos
WHERE dias_desde_ajuizamento < 0;

In [ ]:
%sql
-- Timeline de um processo para conferir visualmente
SELECT
    numero_processo,
    nome_movimento,
    data_movimento,
    dias_desde_ajuizamento,
    dias_desde_ultimo_movimento,
    is_ultimo_movimento
FROM judicial.silver.movimentos
WHERE numero_processo = (SELECT numero_processo FROM judicial.silver.movimentos LIMIT 1)
ORDER BY data_movimento;